# Retinal Disease Classification - Full Training (Google Colab)

This notebook runs **full training** (not quick test) on the APTOS 2019 dataset using the project training pipeline.

**Models trained:** ViT-B/16, ResNet-50, EfficientNet-B4.

## 1. Runtime requirements

1. Set runtime to **GPU** (Runtime -> Change runtime type -> GPU)
2. Make sure your Kaggle account has accepted the APTOS competition rules
3. Upload `kaggle.json` when prompted

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/umerkhan-12/dlp_project.git"
REPO_DIR = Path("/content/dlp_project")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

print(f"Working directory: {Path.cwd()}")

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    raise RuntimeError("GPU is not enabled. Please switch runtime to GPU.")

In [ ]:
import json

# Option A (recommended): paste your Kaggle credentials directly.
# Set both values below.
KAGGLE_USERNAME = "k230798umerkanthi"
KAGGLE_KEY = "KGAT_3df93fc7685a9da792cd51467f399341"

# Option B: leave the values empty and upload kaggle.json when prompted.
if KAGGLE_USERNAME and KAGGLE_KEY:
    kaggle_payload = {"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}
else:
    from google.colab import files
    print("Upload kaggle.json")
    uploaded = files.upload()
    if "kaggle.json" not in uploaded:
        raise FileNotFoundError("kaggle.json was not uploaded")
    kaggle_payload = json.loads(uploaded["kaggle.json"].decode("utf-8"))

if not kaggle_payload.get("username") or not kaggle_payload.get("key"):
    raise ValueError("Kaggle credentials must include both username and key")

os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w", encoding="utf-8") as f:
    json.dump(kaggle_payload, f)
os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)

print("Kaggle API key configured")

In [ ]:
import pandas as pd
import zipfile

data_dir = REPO_DIR / "data" / "raw"
aptos_dir = data_dir / "aptos2019-blindness-detection"
train_csv = aptos_dir / "train.csv"
images_dir = aptos_dir / "train_images"

# Try using the download script first
result = subprocess.run(
    [sys.executable, "scripts/download_data.py", "--dataset", "aptos", "--data-dir", str(data_dir)],
    capture_output=True,
    text=True,
)

print("Download script output:")
print(result.stdout)
if result.returncode != 0:
    print("Download script stderr:")
    print(result.stderr)

# If download script didn't work, try direct Kaggle API call
if not train_csv.exists():
    print("\nTrying direct kaggle download...")
    data_dir.mkdir(parents=True, exist_ok=True)
    os.system(f'kaggle competitions download -c aptos2019-blindness-detection -p "{data_dir}"')
    
    # Extract if zip exists
    zip_path = data_dir / "aptos2019-blindness-detection.zip"
    if zip_path.exists():
        print(f"Extracting {zip_path}...")
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(aptos_dir)
        zip_path.unlink()
        print(f"Extracted to {aptos_dir}")

# Verify
if not train_csv.exists() or not images_dir.exists():
    print(f"\n❌ Dataset verification failed:")
    print(f"  train.csv exists: {train_csv.exists()}")
    print(f"  train_images dir exists: {images_dir.exists()}")
    print(f"  aptos_dir contents: {list(aptos_dir.glob('*')) if aptos_dir.exists() else 'dir does not exist'}")
    raise FileNotFoundError(f"Dataset download failed. Check Kaggle credentials and competition acceptance.")

df = pd.read_csv(train_csv)
image_count = len(list(images_dir.glob("*.png")))
print(f"\n✓ Dataset verified:")
print(f"  train.csv rows: {len(df)}")
print(f"  train_images png count: {image_count}")

In [ ]:
# Optional: persist outputs in Google Drive
USE_GOOGLE_DRIVE = True
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = Path('/content/drive/MyDrive/retinal_disease_full_training')
else:
    OUTPUT_DIR = REPO_DIR / 'experiments'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

TRAIN_PLAN = [
    {"model": "vit_base_patch16_224", "epochs": 100, "batch_size": 16, "lr": "3e-4"},
    {"model": "resnet50", "epochs": 60, "batch_size": 32, "lr": "3e-4"},
    {"model": "efficientnet_b4", "epochs": 60, "batch_size": 20, "lr": "2e-4"},
]
TRAIN_PLAN

In [ ]:
DATA_DIR = REPO_DIR / "data" / "raw" / "aptos2019-blindness-detection"

for item in TRAIN_PLAN:
    cmd = [
        sys.executable, "main.py",
        "--data-dir", str(DATA_DIR),
        "--output-dir", str(OUTPUT_DIR),
        "--model", item["model"],
        "--epochs", str(item["epochs"]),
        "--batch-size", str(item["batch_size"]),
        "--lr", item["lr"],
        "--device", "cuda",
        "--num-workers", "2",
        "--early-stopping", "15",
        "--experiment-name", "colab_full",
    ]
    print("\n" + "=" * 80)
    print("Running:", " ".join(cmd))
    print("=" * 80)
    subprocess.run(cmd, cwd=REPO_DIR, check=True)

print("\nAll model trainings finished.")

In [ ]:
import json
import re
import pandas as pd
import torch

def parse_metric_file(path: Path):
    metrics = {}
    if not path.exists():
        return metrics
    for line in path.read_text().splitlines():
        if ":" not in line:
            continue
        key, value = line.split(":", 1)
        key = key.strip().lower().replace(" ", "_").replace("(", "").replace(")", "")
        value = value.strip()
        if re.fullmatch(r"[-+]?\d*\.?\d+", value):
            metrics[key] = float(value)
    return metrics

rows = []
for exp_dir in sorted([p for p in OUTPUT_DIR.glob("*") if p.is_dir()]):
    config_path = exp_dir / "config.json"
    ckpt_path = exp_dir / "checkpoints" / "best_model.pth"
    metrics_path = exp_dir / "results" / "test_metrics.txt"
    if not config_path.exists():
        continue

    cfg = json.loads(config_path.read_text())
    best_epoch = None
    if ckpt_path.exists():
        ckpt = torch.load(ckpt_path, map_location="cpu")
        best_epoch = int(ckpt.get("epoch", -1)) + 1

    parsed = parse_metric_file(metrics_path)
    rows.append({
        "run": exp_dir.name,
        "model": cfg.get("model"),
        "target_epochs": cfg.get("epochs"),
        "best_epoch": best_epoch,
        "accuracy": parsed.get("accuracy"),
        "balanced_accuracy": parsed.get("balanced_accuracy"),
        "f1_macro": parsed.get("f1_macro"),
        "quadratic_kappa": parsed.get("quadratic_kappa"),
    })

summary_df = pd.DataFrame(rows)
summary_df = summary_df.sort_values(["model", "run"]).reset_index(drop=True)
summary_df

In [ ]:
# Archive all outputs and optionally download
import shutil
from google.colab import files

archive_path = Path('/content/retinal_disease_full_training_artifacts')
zip_file = shutil.make_archive(str(archive_path), 'zip', root_dir=str(OUTPUT_DIR))
print(f"Created archive: {zip_file}")

# Uncomment if you want direct browser download (large files may fail in browser):
# files.download(zip_file)

## Notes

- If Colab disconnects, rerun from the training section; completed runs in `OUTPUT_DIR` are kept.
- For the best stability, keep `USE_GOOGLE_DRIVE=True` so checkpoints and results persist.